In [ ]:
# =========================
# Installs and imports
# =========================
!pip -q install pandas numpy requests lxml tqdm pylatexenc

import os
import re
import time
import ast
import random
import requests
import pandas as pd
import numpy as np

from tqdm.auto import tqdm
from lxml import etree
from pylatexenc.latex2text import LatexNodes2Text
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

random.seed(42)
np.random.seed(42)

PMC_OAI = "https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/"
HEADERS = {
    "User-Agent": "ColabAbstractConclusionBuilder/1.0 (research use)",
    "Accept-Encoding": "gzip, deflate",
}

LATEX2TEXT = LatexNodes2Text()

# =========================
# Config
# =========================
MAX_PMC_RECORDS = 80000
CHECKPOINT_EVERY = 2500

FINAL_PKL = "pmc_abstract_conclusion_data.pkl"
FINAL_CSV = "pmc_abstract_conclusion_data.csv"
CHECKPOINT_PKL = "pmc_abstract_conclusion_data_checkpoint.pkl"
CHECKPOINT_CSV = "pmc_abstract_conclusion_data_checkpoint.csv"
TAGS_CSV = "all_unique_mesh_tags.csv"

# =========================
# Text helpers
# =========================
def norm_text(s: str) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    return re.sub(r"\s+", " ", str(s)).strip()

def node_text(node) -> str:
    if node is None:
        return ""
    try:
        return norm_text(" ".join(node.itertext()))
    except Exception:
        return norm_text(str(node))

def clean_term(s: str) -> str:
    s = norm_text(s)
    s = re.sub(r"\s+", " ", s)
    return s.strip(" ;,./")

def latex_to_plain(tex: str) -> str:
    tex = tex or ""
    tex = re.sub(r"(?<!\\)%.*", "", tex)
    tex = re.sub(
        r"\\begin\{(figure|table|align|equation|thebibliography|bibliography|lstlisting)\}.*?\\end\{\1\}",
        " ",
        tex,
        flags=re.I | re.S,
    )
    try:
        tex = LATEX2TEXT.latex_to_text(tex)
    except Exception:
        pass
    return norm_text(tex)

def remove_simple_citations(text):
    return re.sub(r"\[(?:\s*\d+\s*(?:[-,]\s*\d+\s*)*)\]", " ", text)

def clean_scientific_text(text):
    text = norm_text(text)
    text = remove_simple_citations(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def word_count(text):
    return len(text.split()) if text else 0

def unique_keep_order(items):
    seen = set()
    out = []
    for x in items:
        x = clean_term(x)
        if not x:
            continue
        key = x.lower()
        if key not in seen:
            seen.add(key)
            out.append(x)
    return out

# =========================
# Robust session and harvester
# =========================
def get_robust_session():
    session = requests.Session()
    retry_strategy = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

def pmc_oai_records_robust(max_records=None):
    """
    Harvest full-text PMC records from the official OAI-PMH API.
    Uses set=pmc-open and metadataPrefix=pmc.
    """
    session = get_robust_session()
    collected = 0
    params = {
        "verb": "ListRecords",
        "set": "pmc-open",
        "metadataPrefix": "pmc",
    }

    while True:
        try:
            r = session.get(PMC_OAI, params=params, headers=HEADERS, timeout=180)
            r.raise_for_status()

            root = etree.fromstring(r.content)

            record_nodes = root.xpath('//*[local-name()="record"]')
            for rec in record_nodes:
                yield rec
                collected += 1
                if max_records is not None and collected >= max_records:
                    return

            token_nodes = root.xpath('//*[local-name()="resumptionToken"]')
            token = node_text(token_nodes[0]) if token_nodes else ""
            if not token:
                break

            params = {"verb": "ListRecords", "resumptionToken": token}
            time.sleep(1.0)

        except Exception as e:
            print(f"\nHarvester warning after {collected} records: {e}")
            time.sleep(2.0)
            continue

# =========================
# Extraction helpers
# =========================
CONCL_TITLE_RE = re.compile(
    r"\b(conclusion|conclusions|concluding remarks|concluding statement|summary and conclusion|final remarks)\b",
    flags=re.I,
)

def extract_first_abstract_jats(article_root) -> str:
    abs_nodes = article_root.xpath('.//*[local-name()="abstract"]')
    if not abs_nodes:
        return ""
    return node_text(abs_nodes[0])

def extract_conclusion_jats(article_root) -> str:
    sec_nodes = article_root.xpath('.//*[local-name()="body"]//*[local-name()="sec"]')
    if not sec_nodes:
        sec_nodes = article_root.xpath('.//*[local-name()="sec"]')

    for sec in sec_nodes:
        title_nodes = sec.xpath('./*[local-name()="title"][1]')
        title = node_text(title_nodes[0]) if title_nodes else ""
        if CONCL_TITLE_RE.search(title):
            txt = node_text(sec)
            if len(txt.split()) >= 20:
                return txt
    return ""

def extract_mesh_terms(article):
    terms = []

    for kwd_group in article.xpath('.//*[local-name()="kwd-group"]'):
        for node in kwd_group.xpath(
            './/*[local-name()="kwd"] | '
            './/*[local-name()="compound-kwd"]//*[local-name()="kwd"] | '
            './/*[local-name()="nested-kwd"]//*[local-name()="kwd"]'
        ):
            terms.append(node_text(node))

    for subj_group in article.xpath('.//*[local-name()="subj-group"]'):
        for node in subj_group.xpath('.//*[local-name()="subject"]'):
            terms.append(node_text(node))

    for node in article.xpath('.//*[local-name()="kwd"]'):
        terms.append(node_text(node))

    return unique_keep_order(terms)

def extract_pmc_pair(record_node):
    article_nodes = record_node.xpath('.//*[local-name()="article"]')
    if not article_nodes:
        return None
    art = article_nodes[0]

    pmcid = ""
    pmid = ""

    for node in art.xpath('.//*[local-name()="article-id"]'):
        typ = (node.get("pub-id-type") or "").lower().strip()
        val = node_text(node)
        if not val:
            continue
        if typ in ("pmc", "pmcid") and not pmcid:
            pmcid = val if val.upper().startswith("PMC") else f"PMC{val}"
        elif typ == "pmid" and not pmid:
            pmid = val

    if not pmcid and not pmid:
        header_ids = record_node.xpath('.//*[local-name()="header"]/*[local-name()="identifier"]')
        header_id = node_text(header_ids[0]) if header_ids else ""
        m = re.search(r"(\d+)$", header_id)
        pmcid = f"PMC{m.group(1)}" if m else ""

    source_id = pmcid or pmid
    if not source_id:
        return None

    title_nodes = art.xpath('.//*[local-name()="article-title"]')
    title = node_text(title_nodes[0]) if title_nodes else ""

    abstract = extract_first_abstract_jats(art)
    conclusion = extract_conclusion_jats(art)
    mesh_terms = extract_mesh_terms(art)

    # keep only records that have an abstract AND a detected conclusion
    if not abstract or not conclusion:
        return None

    return {
        "source_domain": "biomedical",
        "source_id": source_id,
        "title": title,
        "abstract": abstract,
        "conclusion": conclusion,
        "mesh_terms": mesh_terms,
        "mesh_terms_str": " | ".join(mesh_terms),
        "mesh_terms_count": len(mesh_terms),
        "abstract_len": len(abstract.split()),
        "conclusion_len": len(conclusion.split()),
    }

# =========================
# Harvest + checkpoint saving
# =========================
pmc_rows = []
seen_keys = set()

def save_checkpoint(rows):
    df = pd.DataFrame(rows)
    if df.empty:
        return
    df = df.drop_duplicates(subset=["source_id", "abstract", "conclusion", "mesh_terms_str"]).reset_index(drop=True)
    df.to_pickle(CHECKPOINT_PKL)
    df.to_csv(CHECKPOINT_CSV, index=False)

# Resume from checkpoint if present
if os.path.exists(CHECKPOINT_PKL):
    try:
        ckpt_df = pd.read_pickle(CHECKPOINT_PKL)
        pmc_rows = ckpt_df.to_dict("records")
        for r in pmc_rows:
            seen_keys.add((r["source_id"], r["abstract"], r["conclusion"], r["mesh_terms_str"]))
        print(f"Loaded checkpoint with {len(pmc_rows)} rows.")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")
        pmc_rows = []
        seen_keys = set()

for rec in tqdm(pmc_oai_records_robust(max_records=MAX_PMC_RECORDS), desc="Harvesting PMC records"):
    row = extract_pmc_pair(rec)
    if row is None:
        continue

    key = (row["source_id"], row["abstract"], row["conclusion"], row["mesh_terms_str"])
    if key in seen_keys:
        continue

    seen_keys.add(key)
    pmc_rows.append(row)

    if len(pmc_rows) % CHECKPOINT_EVERY == 0:
        save_checkpoint(pmc_rows)
        print(f"Checkpoint saved at {len(pmc_rows)} usable rows.")

pmc_df = pd.DataFrame(pmc_rows).drop_duplicates(
    subset=["source_id", "abstract", "conclusion", "mesh_terms_str"]
).reset_index(drop=True)

print("\nPMC extracted usable records:", len(pmc_df))
print(pmc_df.head(10))

# final save
pmc_df.to_pickle(FINAL_PKL)
pmc_df.to_csv(FINAL_CSV, index=False)

print(f"\nSaved final dataset to {FINAL_PKL} and {FINAL_CSV}")

# =========================
# Unique tag extraction
# =========================
def parse_tags(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if isinstance(value, tuple) or isinstance(value, set):
        return [str(x).strip() for x in value if str(x).strip()]
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return []
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple, set)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass
        parts = [p.strip() for p in re.split(r"\s*\|\s*|;\s*|,\s*", s) if p.strip()]
        return parts
    return [str(value).strip()] if str(value).strip() else []

all_tags = []
for x in pmc_df["mesh_terms"]:
    all_tags.extend(parse_tags(x))

unique_tags = sorted(set(all_tags), key=lambda s: s.lower())
tags_df = pd.DataFrame({"raw_tag": unique_tags})
tags_df.to_csv(TAGS_CSV, index=False)

print(f"Saved unique tags to {TAGS_CSV}")
print("Unique tags:", len(tags_df))

# Optional preview
view_cols = [
    "source_id", "title", "mesh_terms_count",
    "mesh_terms_str", "abstract_len", "conclusion_len",
    "abstract", "conclusion"
]
print("\nPreview:")
display(pmc_df[view_cols].head(10))

In [ ]:
# =========================================================
# Full pair-level dataset creation with proper filtering
# =========================================================

!pip -q install pandas numpy

import ast
import os
import re
import math
import random
import unicodedata
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = random.Random(SEED)

# =========================================================
# Inputs
# =========================================================
# Assumes:
#   - pmc_df is already loaded in memory
#   - /content/all_unique_mesh_tags.csv exists
#
# pmc_df should contain at least:
#   source_id, title, abstract, conclusion, mesh_terms

pmc_df = pd.read_csv("/content/pmc_abstract_conclusion_data.csv")
tags_df = pd.read_csv("/content/all_unique_mesh_tags.csv")
tags_df["raw_tag"] = tags_df["raw_tag"].astype(str).str.strip()

print("pmc_df columns:", list(pmc_df.columns))
print("Unique tags loaded:", len(tags_df))

# =========================================================
# Filtering settings
# =========================================================
MIN_ABS_WORDS = 20
MIN_CONCL_WORDS = 10
MAX_ABS_WORDS_MODEL = 300
MAX_CONCL_WORDS_MODEL = 300
MAX_LENGTH_RATIO = 12.0

NEG_POS_RATIO = 65 / 35   # negative : positive

# =========================================================
# Text helpers
# =========================================================
def normalize_text(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    text = text.replace("\u00ad", "").replace("\u200b", "").replace("\ufeff", "")
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

def remove_simple_citations(text):
    return re.sub(r"\[(?:\s*\d+\s*(?:[-,]\s*\d+\s*)*)\]", " ", text)

def clean_scientific_text(text):
    text = normalize_text(text)
    text = remove_simple_citations(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def word_count(text):
    return len(text.split()) if text else 0

def cap_words(text, max_words):
    words = (text or "").split()
    return " ".join(words[:max_words]).strip()

def parse_tags(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if isinstance(value, (tuple, set)):
        return [str(x).strip() for x in value if str(x).strip()]
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return []
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple, set)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass
        parts = [p.strip() for p in re.split(r"\s*\|\s*|;\s*|,\s*", s) if p.strip()]
        return parts
    return [str(value).strip()] if str(value).strip() else []

def normalize_tag_for_matching(tag):
    if tag is None:
        return ""
    t = normalize_text(tag)

    # remove PubChem noise
    t = re.sub(r"\(PubChem CID:\s*\d+\)", "", t, flags=re.I)
    t = re.sub(r"PubChem CID:\s*\d+", "", t, flags=re.I)

    greek_map = {
        "α": "alpha", "β": "beta", "γ": "gamma", "δ": "delta", "ε": "epsilon",
        "κ": "kappa", "μ": "mu", "ω": "omega", "σ": "sigma", "τ": "tau",
        "θ": "theta", "λ": "lambda", "π": "pi"
    }
    for k, v in greek_map.items():
        t = t.replace(k, v)

    t = re.sub(r"\s+", " ", t)
    return t.strip(" ,;./-")

def unique_keep_order(items):
    seen = set()
    out = []
    for x in items:
        x = normalize_text(x)
        if not x:
            continue
        key = x.lower()
        if key not in seen:
            seen.add(key)
            out.append(x)
    return out

# =========================================================
# Grouping rules
# =========================================================
GROUP_RULES = [
    ("Oncology", [
        r"\bcancer\b", r"\bneoplasm", r"\btumou?r\b", r"\blymphoma", r"\bleukemia\b",
        r"\bmelanoma\b", r"\bsarcoma\b", r"\bcarcinoma\b", r"\boncolog\b", r"breast cancer",
        r"prostate cancer", r"lung cancer", r"gastric cancer", r"cervical cancer"
    ]),
    ("Cardiology/Vascular", [
        r"coronary", r"cardio", r"heart", r"myocard", r"vascular", r"blood pressure",
        r"hypertension", r"stroke", r"arrhythm", r"angi", r"atheroscl"
    ]),
    ("Neurology", [
        r"brain", r"neuro", r"neuron", r"cerebral", r"spinal", r"parkinson", r"alzheimer",
        r"epilep", r"migraine", r"dementia"
    ]),
    ("Psychiatry/Psychology", [
        r"psych", r"depression", r"anxiety", r"behavior", r"cognitive", r"mental", r"stress",
        r"delirium", r"psychosocial"
    ]),
    ("Immunology/Infectious Disease", [
        r"immun", r"virus", r"viral", r"bacter", r"infect", r"antigen", r"antibody",
        r"lymph", r"cytokine", r"interleukin", r"\bhiv\b", r"hepatitis", r"parasit", r"fung"
    ]),
    ("Pulmonology/Respiratory", [
        r"lung", r"respirat", r"asthma", r"bronch", r"pulmonary", r"ventilation", r"hypoxia", r"airway"
    ]),
    ("Endocrinology/Metabolism", [
        r"diabetes", r"endocr", r"insulin", r"obesity", r"metabol", r"thyroid",
        r"cholesterol", r"lipid", r"glucose", r"adip"
    ]),
    ("Gastroenterology/Hepatology", [
        r"liver", r"hepatic", r"gastro", r"intestinal", r"stomach", r"colon", r"bowel",
        r"esoph", r"pancre", r"gastric", r"ileum"
    ]),
    ("Nephrology/Urology", [
        r"kidney", r"renal", r"urinary", r"bladder", r"prostate", r"urine", r"nephro", r"urolog"
    ]),
    ("Obstetrics/Gynecology/Pediatrics", [
        r"pregnan", r"conceptus", r"fetus", r"fetal", r"maternal", r"women", r"child", r"children",
        r"infant", r"neonat", r"breast", r"gynec", r"obstet", r"female"
    ]),
    ("Musculoskeletal/Rheumatology", [
        r"bone", r"joint", r"muscle", r"rheumat", r"arthritis", r"osteo", r"myo", r"tendon", r"spine"
    ]),
    ("Genetics/Molecular Biology", [
        r"\bgene\b", r"\bgenom", r"\brna\b", r"\bdna\b", r"mutation", r"transcription",
        r"translation", r"sequence", r"polymorphism", r"microrna", r"\bmir-", r"recombinant",
        r"protein", r"enzyme", r"receptor", r"kinase", r"signaling"
    ]),
    ("Cell Biology/Histology", [
        r"\bcell\b", r"tissue", r"epithe", r"histolog", r"cyto", r"cell culture", r"cell line",
        r"endothelial", r"stem cell", r"differentiation"
    ]),
    ("Pharmacology/Chemicals", [
        r"\bdrug\b", r"therapy", r"chemotherap", r"antiviral", r"pharmac", r"\bagonist\b",
        r"\binhibitor\b", r"pubchem cid", r"compound", r"benzodiazepine", r"colchicine", r"camptothecin"
    ]),
    ("Diagnostics/Imaging/Methods", [
        r"diagnos", r"radiolog", r"ultrason", r"\bmri\b", r"\bct\b", r"assay", r"marker",
        r"biomarker", r"flow cytometry", r"immunohistochem", r"gc-ms", r"nmr", r"imaging", r"spectroscop"
    ]),
    ("Procedures/Treatment", [
        r"surgery", r"transplant", r"rehabilitation", r"ventilation", r"procedur", r"intervention", r"treatment"
    ]),
    ("Epidemiology/Study Design", [
        r"case control", r"cohort", r"random", r"prevalence", r"incidence", r"survival",
        r"analysis", r"statistics", r"controlled study"
    ]),
    ("Health Services/Nursing/Rehab", [
        r"health care", r"health service", r"home nursing", r"rehabilitation", r"ambulatory", r"nursing"
    ]),
    ("Microbiology/Virology/Parasitology", [
        r"saccharomyces", r"escherichia", r"staphyl", r"strept", r"fung", r"parasit", r"viral",
        r"bacterial", r"microbiolog", r"p\. falciparum"
    ]),
    ("Veterinary/Animal/Plant/Botany", [
        r"primates", r"murine", r"mouse", r"rat", r"canine", r"feline", r"animal",
        r"plants?", r"vitis vinifera", r"zebrafish", r"insects", r"species"
    ]),
    ("Public Health/Population/Social", [
        r"population", r"public health", r"workers", r"blue-collar", r"geography",
        r"community", r"survey", r"epidemiolog"
    ]),
    ("Computational/Bioinformatics/Statistics", [
        r"machine learning", r"clustering", r"bioinformat", r"algorithm", r"comput", r"blast",
        r"network", r"model", r"simulation", r"data mining"
    ]),
    ("Anatomy/Developmental", [
        r"anatom", r"corneal", r"basement membrane", r"ileum", r"ovotestis", r"organ"
    ]),
]

def suggest_group(tag):
    t = normalize_tag_for_matching(tag).lower()
    if re.fullmatch(r"\d+(\.\d+)?", t):
        return "Unmapped"

    for group, patterns in GROUP_RULES:
        for pat in patterns:
            if re.search(pat, t, flags=re.I):
                return group
    return "Unmapped"

# Optional manual overrides if you create manual_tag_grouping.csv
manual_map = None
if os.path.exists("manual_tag_grouping.csv"):
    manual_map = pd.read_csv("manual_tag_grouping.csv")
    manual_map["raw_tag"] = manual_map["raw_tag"].astype(str).str.strip()
    print("Loaded manual_tag_grouping.csv")
else:
    print("No manual_tag_grouping.csv found; using heuristic suggestions.")

def map_tag_to_group(tag):
    tag = normalize_tag_for_matching(tag)
    if not tag:
        return None

    if manual_map is not None:
        hit = manual_map[manual_map["raw_tag"].str.lower() == tag.lower()]
        if not hit.empty:
            if "final_group" in hit.columns:
                fg = str(hit.iloc[0].get("final_group", "")).strip()
                if fg:
                    return fg
            if "suggested_group" in hit.columns:
                sg = str(hit.iloc[0].get("suggested_group", "")).strip()
                if sg:
                    return sg

    return suggest_group(tag)

def assign_primary_subdomain(tag_list):
    tags = parse_tags(tag_list)
    if not tags:
        return "Other/Unmapped", ["Other/Unmapped"], []

    counts = {}
    mapped_terms = []

    for t in tags:
        g = map_tag_to_group(t)
        if g and g != "Unmapped":
            counts[g] = counts.get(g, 0) + 1
            mapped_terms.append((t, g))

    if not counts:
        return "Other/Unmapped", ["Other/Unmapped"], mapped_terms

    max_count = max(counts.values())
    winners = sorted([k for k, v in counts.items() if v == max_count])
    primary = winners[0]
    all_groups = sorted(counts.keys(), key=lambda k: (-counts[k], k))
    return primary, all_groups, mapped_terms

# Save tag review sheet
tags_df["suggested_group"] = tags_df["raw_tag"].map(suggest_group)
tags_df.to_csv("mesh_tag_review_with_suggestions.csv", index=False)
print("Saved mesh_tag_review_with_suggestions.csv")

# =========================================================
# Prepare usable papers
# =========================================================
def prepare_usable_rows(df):
    df = df.copy()

    required = ["source_id", "title", "abstract", "conclusion", "mesh_terms"]
    for col in required:
        if col not in df.columns:
            df[col] = None

    df["title"] = df["title"].map(normalize_text)
    df["abstract_clean"] = df["abstract"].map(clean_scientific_text)
    df["conclusion_clean"] = df["conclusion"].map(clean_scientific_text)

    df["abstract_clean"] = df["abstract_clean"].replace("", np.nan)
    df["conclusion_clean"] = df["conclusion_clean"].replace("", np.nan)
    df = df.dropna(subset=["abstract_clean", "conclusion_clean"]).copy()

    df["abstract_wc"] = df["abstract_clean"].map(word_count)
    df["conclusion_wc"] = df["conclusion_clean"].map(word_count)

    # much looser filtering than before
    df = df[
        (df["abstract_wc"] >= MIN_ABS_WORDS) &
        (df["conclusion_wc"] >= MIN_CONCL_WORDS)
    ].copy()

    df["length_ratio"] = np.maximum(df["abstract_wc"], df["conclusion_wc"]) / np.maximum(
        1, np.minimum(df["abstract_wc"], df["conclusion_wc"])
    )
    df = df[df["length_ratio"] <= MAX_LENGTH_RATIO].copy()

    df["abstract_model"] = df["abstract_clean"].map(lambda x: cap_words(x, MAX_ABS_WORDS_MODEL))
    df["conclusion_model"] = df["conclusion_clean"].map(lambda x: cap_words(x, MAX_CONCL_WORDS_MODEL))

    sub_info = df["mesh_terms"].apply(assign_primary_subdomain)
    df["primary_subdomain"] = sub_info.apply(lambda x: x[0])
    df["all_subdomains"] = sub_info.apply(lambda x: x[1])
    df["mapped_terms"] = sub_info.apply(lambda x: x[2])

    # keep unmapped temporarily; we will remove them before pairing
    df["primary_subdomain"] = df["primary_subdomain"].fillna("Other/Unmapped")
    df["all_subdomains"] = df["all_subdomains"].apply(
        lambda x: x if isinstance(x, list) and len(x) > 0 else ["Other/Unmapped"]
    )

    df = df.drop_duplicates(subset=["source_id"]).reset_index(drop=True)
    return df

usable_df = prepare_usable_rows(pmc_df)

print("\nUsable records after filtering:", len(usable_df))
print("Primary subdomain counts:")
print(usable_df["primary_subdomain"].value_counts().head(20))

# =========================================================
# Remove unassigned records BEFORE pairing
# =========================================================
def is_valid_subdomain(x):
    if x is None:
        return False
    s = str(x).strip()
    return s not in {"Other/Unmapped", "Unmapped", "", "nan", "None"}

usable_df = usable_df[usable_df["primary_subdomain"].apply(is_valid_subdomain)].reset_index(drop=True)
usable_df["all_subdomains"] = usable_df["all_subdomains"].apply(
    lambda x: x if isinstance(x, list) and any(is_valid_subdomain(v) for v in x) else ["Other/Unmapped"]
)

print("\nUsable records after dropping unassigned:", len(usable_df))
print("Primary subdomain counts after drop:")
print(usable_df["primary_subdomain"].value_counts().head(20))

# =========================================================
# Balanced pair generation
# Positive = same paper
# Negative = different primary subdomain (strict first, relaxed if needed)
# =========================================================
def has_disjoint_subdomains(a_row, b_row):
    a_set = set(a_row["all_subdomains"]) if isinstance(a_row["all_subdomains"], list) and a_row["all_subdomains"] else {a_row["primary_subdomain"]}
    b_set = set(b_row["all_subdomains"]) if isinstance(b_row["all_subdomains"], list) and b_row["all_subdomains"] else {b_row["primary_subdomain"]}
    return a_set.isdisjoint(b_set)

def build_balanced_dataset(df):
    records = df.to_dict("records")

    # Positive pairs: one per paper, same paper
    pos_rows = []
    for row in records:
        pos_rows.append({
            "pair_type": "positive_same_paper",
            "label": 1,
            "abstract_paper_id": row["source_id"],
            "conclusion_paper_id": row["source_id"],
            "abstract_subdomain": row["primary_subdomain"],
            "conclusion_subdomain": row["primary_subdomain"],
            "abstract_title": row["title"],
            "conclusion_title": row["title"],
            "abstract_raw": row["abstract_clean"],
            "conclusion_raw": row["conclusion_clean"],
            "abstract_model": row["abstract_model"],
            "conclusion_model": row["conclusion_model"],
            "abstract_wc": row["abstract_wc"],
            "conclusion_wc": row["conclusion_wc"],
            "abstract_subdomains_all": row["all_subdomains"],
            "conclusion_subdomains_all": row["all_subdomains"],
        })

    positive_count = len(pos_rows)
    target_neg = math.ceil(positive_count * NEG_POS_RATIO)

    group_to_rows = {}
    for row in records:
        group_to_rows.setdefault(row["primary_subdomain"], []).append(row)

    all_groups = list(group_to_rows.keys())
    if len(all_groups) < 2:
        raise ValueError("Need at least two valid subdomains to build negative pairs.")

    neg_rows = []
    seen_keys = set()
    attempts = 0
    max_attempts = target_neg * 300

    # strict negatives first
    while len(neg_rows) < target_neg and attempts < max_attempts:
        attempts += 1

        a = rng.choice(records)
        candidate_groups = [g for g in all_groups if g != a["primary_subdomain"]]
        if not candidate_groups:
            continue

        b_group = rng.choice(candidate_groups)
        b = rng.choice(group_to_rows[b_group])

        if a["source_id"] == b["source_id"]:
            continue

        a_set = set(a["all_subdomains"]) if isinstance(a["all_subdomains"], list) and a["all_subdomains"] else {a["primary_subdomain"]}
        b_set = set(b["all_subdomains"]) if isinstance(b["all_subdomains"], list) and b["all_subdomains"] else {b["primary_subdomain"]}

        if not a_set.isdisjoint(b_set):
            continue

        key = (a["source_id"], b["source_id"], 0)
        if key in seen_keys:
            continue
        seen_keys.add(key)

        neg_rows.append({
            "pair_type": "negative_different_domain",
            "label": 0,
            "abstract_paper_id": a["source_id"],
            "conclusion_paper_id": b["source_id"],
            "abstract_subdomain": a["primary_subdomain"],
            "conclusion_subdomain": b["primary_subdomain"],
            "abstract_title": a["title"],
            "conclusion_title": b["title"],
            "abstract_raw": a["abstract_clean"],
            "conclusion_raw": b["conclusion_clean"],
            "abstract_model": a["abstract_model"],
            "conclusion_model": b["conclusion_model"],
            "abstract_wc": a["abstract_wc"],
            "conclusion_wc": b["conclusion_wc"],
            "abstract_subdomains_all": a["all_subdomains"],
            "conclusion_subdomains_all": b["all_subdomains"],
        })

    # relaxed fill if needed
    if len(neg_rows) < target_neg:
        print(f"Warning: strict negatives produced {len(neg_rows)}/{target_neg}. Filling the rest more loosely.")
        attempts2 = 0
        max_attempts2 = (target_neg - len(neg_rows)) * 300

        while len(neg_rows) < target_neg and attempts2 < max_attempts2:
            attempts2 += 1

            a = rng.choice(records)
            candidate_groups = [g for g in all_groups if g != a["primary_subdomain"]]
            if not candidate_groups:
                continue

            b_group = rng.choice(candidate_groups)
            b = rng.choice(group_to_rows[b_group])

            if a["source_id"] == b["source_id"]:
                continue

            key = (a["source_id"], b["source_id"], 0)
            if key in seen_keys:
                continue
            seen_keys.add(key)

            neg_rows.append({
                "pair_type": "negative_different_domain",
                "label": 0,
                "abstract_paper_id": a["source_id"],
                "conclusion_paper_id": b["source_id"],
                "abstract_subdomain": a["primary_subdomain"],
                "conclusion_subdomain": b["primary_subdomain"],
                "abstract_title": a["title"],
                "conclusion_title": b["title"],
                "abstract_raw": a["abstract_clean"],
                "conclusion_raw": b["conclusion_clean"],
                "abstract_model": a["abstract_model"],
                "conclusion_model": b["conclusion_model"],
                "abstract_wc": a["abstract_wc"],
                "conclusion_wc": b["conclusion_wc"],
                "abstract_subdomains_all": a["all_subdomains"],
                "conclusion_subdomains_all": b["all_subdomains"],
            })

    if len(neg_rows) < target_neg:
        raise RuntimeError(
            f"Could only build {len(neg_rows)} negatives out of {target_neg}."
        )

    pair_df = pd.DataFrame(pos_rows + neg_rows)

    # better shuffle
    pair_df = pair_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    pair_df["input_text"] = (
        "Abstract: " + pair_df["abstract_model"] +
        " [SEP] Conclusion: " + pair_df["conclusion_model"]
    )

    keep_cols = [
        "pair_type", "label",
        "abstract_paper_id", "conclusion_paper_id",
        "abstract_subdomain", "conclusion_subdomain",
        "abstract_subdomains_all", "conclusion_subdomains_all",
        "abstract_title", "conclusion_title",
        "abstract_raw", "conclusion_raw",
        "abstract_model", "conclusion_model",
        "abstract_wc", "conclusion_wc",
        "input_text"
    ]
    pair_df = pair_df[keep_cols].reset_index(drop=True)
    return pair_df

pair_out = build_balanced_dataset(usable_df)

# =========================================================
# Final counts
# =========================================================
print("\nFinal counts:")
print("Usable PMC records:", len(usable_df))
print("Positive pairs:", (pair_out["label"] == 1).sum())
print("Negative pairs:", (pair_out["label"] == 0).sum())
print("Total pair records:", len(pair_out))

print("\nLabel distribution:")
print(pair_out["label"].value_counts())

print("\nPair type distribution:")
print(pair_out["pair_type"].value_counts())

# =========================================================
# Subdomain summary
# =========================================================
subdomain_summary = (
    pair_out.groupby(["abstract_subdomain", "label"])
    .size()
    .reset_index(name="count")
)

subdomain_pivot = subdomain_summary.pivot(
    index="abstract_subdomain",
    columns="label",
    values="count"
).fillna(0).astype(int)

subdomain_pivot.columns = ["negative_pairs" if c == 0 else "positive_pairs" for c in subdomain_pivot.columns]
subdomain_pivot["total_pairs"] = subdomain_pivot.sum(axis=1)
subdomain_pivot = subdomain_pivot.sort_values("total_pairs", ascending=False)

print("\nSubdomain pair summary:")
print(subdomain_pivot)

# =========================================================
# Save
# =========================================================
pair_out.to_csv("pmc_pair_dataset_balanced_clean.csv", index=False)
pair_out.to_pickle("pmc_pair_dataset_balanced_clean.pkl")
subdomain_pivot.to_csv("subdomain_pair_summary.csv")

print("\nSaved:")
print("- pmc_pair_dataset_balanced_clean.csv")
print("- pmc_pair_dataset_balanced_clean.pkl")
print("- subdomain_pair_summary.csv")

print("\nPreview:")
print(pair_out.head(10))

In [ ]:
combined_domains = pd.concat([
    pair_out["abstract_subdomain"],
    pair_out["conclusion_subdomain"]
]).astype(str).str.strip()

print(combined_domains.value_counts())

In [ ]:
import zipfile
import os

# ------------------ define files ------------------
files_to_zip = ["pmc_pair_dataset_balanced_clean.csv", "subdomain_pair_summary.csv"]
zip_filename = "pmc_pair_dataset.zip"

# ------------------ create zip ------------------
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for f in files_to_zip:
        if os.path.exists(f):
            zipf.write(f, arcname=f)  # store file with original name inside zip

print(f"✅ Dataset and summary table compressed into: {zip_filename}")